# 5 — DeBERTa / NLI faithfulness baseline

Thesis **baseline** filter: fine-tune DeBERTa (`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`) as a binary faithfulness classifier (premise = context, hypothesis = answer), compared against zero-shot NLI.

**Protocol**
- Dataset: `data/labeled_merged.csv`
- Leakage-safe base-ID split (`asqa_X` / `asqa_X_hallu` stay together)
- `test_size=0.2`, `random_state=42` → freeze `data/labeled_merged_test.csv`
- Val carved from remaining train for min-FPR threshold @ recall ≥ 0.70
- Train DeBERTa **once** by default (`n_runs: 1`); same frozen split/seed
- Compare: No Filter / NLI zero-shot / DeBERTa

Heavy lifting lives in `src/filtering/` and `scripts/run_deberta_nli_baseline.py`. This notebook is the interactive driver.


## 1. Setup

In [1]:
%cd /kaggle/working
!rm -rf ragas-evaluation
!git clone https://github.com/Htaaxx/ragas-evaluation.git
%cd /kaggle/working/ragas-evaluation
!pwd
!git branch --show-current
!ls notebooks

/kaggle/working
Cloning into 'ragas-evaluation'...
remote: Enumerating objects: 1454, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 1454 (delta 57), reused 77 (delta 36), pack-reused 1337 (from 1)
Receiving objects: 100% (1454/1454), 41.12 MiB | 23.54 MiB/s, done.
Resolving deltas: 100% (830/830), done.
/kaggle/working/ragas-evaluation
/kaggle/working/ragas-evaluation
main
1_synthetic-data.ipynb		    3.2_filter-training.ipynb
2_rag-asqa-baseline.ipynb	    4_llm-judge-filter.ipynb
3.1_ragas-feature-extraction.ipynb  5_deberta_nli_baseline.ipynb


In [2]:
!git pull origin main

From https://github.com/Htaaxx/ragas-evaluation
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
!pip install datasets

In [4]:
from __future__ import annotations

import os
import sys
from pathlib import Path
import pandas as pd

# ========== 1) SET THIS IF NEEDED ==========
# On Kaggle (clone into /kaggle/working/ragas-evaluation):
REPO_ROOT = Path("/kaggle/working/ragas-evaluation")
# On local machine (notebook inside notebooks/):
if not (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file():
    REPO_ROOT = Path("..").resolve()

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
assert (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file(), (
    "Script not found. Fix REPO_ROOT. Expected: "
    f"{REPO_ROOT}/scripts/run_deberta_nli_baseline.py"
)

# ========== 2) Project imports ==========
# If error says: partially initialized module 'torch' (circular import)
# -> menu: Restart session / Restart kernel, then run Setup again.

from src.filtering.config_loader import load_yaml, resolve_path
from src.filtering.data_split import load_and_split, to_base_id
from src.filtering.learned_filter import (
    _extract_top1_context,
    _truncation_collision_diagnostic,
    overfit_sanity_check,
)
from transformers import AutoTokenizer

CFG_PATH = REPO_ROOT / "configs" / "experiments" / "filter_training.yaml"
cfg = load_yaml(str(CFG_PATH))
data_cfg = cfg["data"]
print("OK — ready. n_runs =", cfg.get("n_runs", 1))


REPO_ROOT = /kaggle/working/ragas-evaluation
OK — ready. n_runs = 1


## 2. Leakage-safe split + freeze test CSV

Same intent as:
```python
train_test_split(df, test_size=0.2, random_state=42)
```
but split on **base IDs** so correct/hallu pairs never cross the train–test boundary.

In [5]:
train_df, val_df, test_df = load_and_split(
    csv_path=str(resolve_path(data_cfg["labeled_csv"])),
    test_ratio=data_cfg["test_ratio"],
    val_ratio=data_cfg["val_ratio"],
    seed=data_cfg["seed"],
    test_csv_path=str(resolve_path(data_cfg["test_csv"])) if data_cfg.get("test_csv") else None,
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: n={len(split)} pos={(split.label==1).sum()} neg={(split.label==0).sum()} bases={split.id.map(to_base_id).nunique()}")
print("OK: no base-ID leakage across splits")
print("Frozen test CSV:", resolve_path(data_cfg["test_csv"]))

print(f'Shape: {test_df.shape}')
print(f'Columns: {list(test_df.columns)}')
display(test_df)

# count sample per dataset
dataset_counts = test_df['dataset'].value_counts()
print("Sample counts per dataset:")
display(dataset_counts)



train: n=6264 pos=3122 neg=3142 bases=3785
val: n=1580 pos=796 neg=784 bases=947
test: n=1962 pos=985 neg=977 bases=1759
OK: no base-ID leakage across splits
Frozen test CSV: /kaggle/working/ragas-evaluation/data/labeled_merged_test.csv
Shape: (1962, 6)
Columns: ['id', 'question', 'context', 'answer', 'label', 'dataset']


,id,question,context,answer,label,dataset
0,asqa_2288_hallu,When did the first indian expedition to antarc...,- (Indian Antarctic Program) The first Indian ...,The first Indian expedition led by an Indian t...,0,asqa
1,14_pos,"When did the Inter Expo Center in Sofia, Bulga...",The Inter Expo Center (IEC) is a multi-purpose...,"The Inter Expo Center in Sofia, Bulgaria opene...",1,wikieval
2,asqa_4328,When was the first mobile phone text message s...,- (QA_1) When was the first mobile phone text ...,SMS messaging was used for the first time on 3...,1,asqa
3,asqa_3065,Who sings you'll be a woman soon?,"- (Girl, You'll Be a Woman Soon) Girl, You'll ...","Girl, You'll Be a Woman Soon is a song written...",1,asqa
4,asqa_3132,When does hotel transylvania part 3 come out?,- (Hotel Transylvania 3: Summer Vacation) Hote...,"Hotel Transylvania 3, released internationally...",1,asqa
...,...,...,...,...,...,...
1957,asqa_31,What is the new tallest building in san franci...,- (List of tallest buildings in San Francisco)...,"In 1967, the tallest building in San Francisco...",1,asqa
1958,asqa_4191_hallu,Who won the battle of siege of leningrad?,- (Siege of Leningrad) Though the siege of Len...,Though the siege of Leningrad lasted for 900 d...,0,asqa
1959,asqa_3965_hallu,Where did the archaeological department of ind...,- (Archaeological Survey of India) The Archaeo...,The Archaeological Survey of India is an India...,0,asqa
1960,asqa_1360,Who invaded south korea and tried to take over...,- (QA_1) Who invaded south korea and tried to ...,Many countries have invaded South Korea to try...,1,asqa


Sample counts per dataset:


dataset
asqa        1737
msmarco      207
wikieval      18
Name: count, dtype: int64

## 3. Pre-training gates

Run in order: label check → pair spot-check → truncation diagnostic → overfit sanity check.

In [6]:
# Gate 1 — label mapping
assert set(train_df["label"].unique()) <= {0, 1}
pos = train_df[train_df.label == 1].iloc[0]
neg = train_df[train_df.id.map(to_base_id) == to_base_id(pos.id)]
neg = neg[neg.label == 0].iloc[0]
print("label==1 (correct) id:", pos.id)
print("label==0 (hallu)   id:", neg.id)
print("question:", pos.question[:120])
print("correct answer:", str(pos.answer)[:160])
print("hallu answer:  ", str(neg.answer)[:160])

label==1 (correct) id: asqa_1
label==0 (hallu)   id: asqa_1_hallu
question: Who won the 2016 ncaa football national championship?
correct answer: The 2015 - 2016 season's ncaa national football championship game was played between the Clemson Tigers and the Alabama Crimson Tide on January 11, 2016. The Al
hallu answer:   The 2015 - 2016 season's NCAA national football championship game was played between the Clemson Tigers and the Alabama Crimson Tide on January 11, 2016. The Cl


In [7]:
# Gate 2 — five paired spot-checks
shown = 0
for base, group in train_df.groupby(train_df.id.map(to_base_id)):
    if not ((group.label == 1).any() and (group.label == 0).any()):
        continue
    c = group[group.label == 1].iloc[0]
    h = group[group.label == 0].iloc[0]
    print("=" * 60)
    print("base:", base)
    print("Q:", c.question[:100])
    print("ctx:", _extract_top1_context(c.context)[:160])
    print("OK :", str(c.answer)[:120])
    print("BAD:", str(h.answer)[:120])
    shown += 1
    if shown >= 5:
        break

base: 10000
Q: define express included contract terms
ctx: [P1] Source: http://definitions.uslegal.com/e/express-contract/\nExpress Contract Law & Legal Definition. An express contract is a contract whose terms the part
OK : An express contract is a contract whose terms the parties have explicitly set out. This is also termed as special contra
BAD: An express included contract term refers to a provision within a contract that is not only explicitly stated by the part
base: 10163
Q: difference between greek and roman culture
ctx: [P1] Source: http://www.differencebetween.net/miscellaneous/religion-miscellaneous/difference-between-greek-gods-and-roman-gods/\n1. There’s a difference betwee
OK : Romans lived in the Roman Empire since as early as 8th century BC, the Greeks lived in Greece during 8th Century to 6th 
BAD: The Romans were primarily influenced by Egyptian culture, which shaped their entire way of life and governance. Unlike t
base: 10274
Q: what does a electric battery do
ctx: 

In [8]:
# Gate 3 — truncation collision (should be < 5%)
from src.filtering.config_loader import load_config_section, FILTERING_CONFIG

lf_cfg = load_config_section(FILTERING_CONFIG, "learned_filter")
tok = AutoTokenizer.from_pretrained(lf_cfg["model_name"])
contexts = [_extract_top1_context(c) for c in train_df["context"].tolist()]
rate = _truncation_collision_diagnostic(
    tok, train_df, contexts, max_length=lf_cfg.get("max_length", 512), n=200,
)
print(f"truncation collision rate = {rate:.3f}")
assert rate <= 0.05, "Raise max_length — discriminative answer tokens are being truncated"

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

truncation collision rate = 0.000


In [9]:
# Gate 4 — HARD GATE: overfit 16 pairs (32 samples)
gate = overfit_sanity_check(train_df, n_pairs=16, epochs=50)
print(gate)
assert gate["train_f1"] >= 0.95, "Overfit gate failed — do not scale to full training"

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Step,Training Loss
1,1.435160
2,1.004130
3,1.411295
4,1.201530
5,1.368718
6,1.145471
7,1.080118
8,1.725026
9,1.208626
10,1.498900


{'train_f1': 0.9696969696969697, 'train_precision': 0.9411764705882353, 'train_recall': 1.0, 'train_fpr': 0.0625, 'n_samples': 32, 'tp': 16, 'tn': 15, 'fp': 1, 'fn': 0}


## 4. Train

1. **Restart kernel**
2. Run **Setup** (section 1)
3. Run gates (section 3) optional but recommended
4. Run the next cell

Results: `results/deberta_nli/summary_classification_report.csv`


In [ ]:
# ========== TRAIN (DeBERTa + NLI) ==========
# Restart kernel, run Setup cell above, then this cell.

import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ragas-evaluation")
if not (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file():
    REPO_ROOT = Path("..").resolve()

script = REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py"
cfg = REPO_ROOT / "configs" / "experiments" / "filter_training.yaml"

print("Running:", script)
print("Config :", cfg)
subprocess.check_call(
    [
        sys.executable,
        str(script),
        "--config",
        str(cfg),
        "--skip-overfit-gate",
    ],
    cwd=str(REPO_ROOT),
)
print("Done. See results/deberta_nli/summary_classification_report.csv")


Running: /kaggle/working/ragas-evaluation/scripts/run_deberta_nli_baseline.py
Config : /kaggle/working/ragas-evaluation/configs/experiments/filter_training.yaml


INFO:src.filtering.data_split:Loaded 9806 samples from /kaggle/working/ragas-evaluation/data/labeled_merged.csv
INFO:src.filtering.data_split:Using frozen test set /kaggle/working/ragas-evaluation/data/labeled_merged_test.csv (1962 rows); remaining for train/val: 7844
INFO:src.filtering.data_split:Train: 6264 samples (pos=3122, neg=3142)
INFO:src.filtering.data_split:Val: 1580 samples (pos=796, neg=784)
INFO:src.filtering.data_split:Test: 1962 samples (pos=985, neg=977)
INFO:__main__:Eval holdout: 1962 rows from data/labeled_merged_test.csv
INFO:__main__:=== DeBERTa run 1/1 ===
INFO:src.filtering.learned_filter:FP32 mode enforced (fp16=False)
INFO:src.filtering.learned_filter:Extracting top-1 context (premise) for faithfulness training
INFO:src.filtering.learned_filter:Context extracted: train=6264/6264, val=1580/1580 non-empty
INFO:src.filtering.learned_filter:DIAGNOSTIC [train]: labels={0: 3142, 1: 3122}, NaN(label=0, answer=0), label_dtype=int64
INFO:src.filtering.learned_filter:DIA

In [ ]:
summary_path = resolve_path(cfg["results_dir"]) / "summary.json"
report_path = resolve_path(cfg["results_dir"]) / "summary_classification_report.csv"

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("n_train/val/test:", summary["n_train"], summary["n_val"], summary["n_test"])
    print("\n=== comparison_table ===")
    print(json.dumps(summary.get("comparison_table"), indent=2))
    if "deberta_mean_std" in summary:
        print(pd.DataFrame(summary["deberta_mean_std"]).T)
else:
    print("No summary yet at", summary_path)
    print("Run: python scripts/run_deberta_nli_baseline.py")

# Same schema as RAGAS / LLM-judge summaries:
# dataset,num_samples,accepted,acceptance_rate,accuracy,precision,recall,f1,roc_auc
if report_path.exists():
    report_df = pd.read_csv(report_path)
    print("\n=== summary_classification_report (DeBERTa mean over runs) ===")
    display(report_df)
else:
    print("\nNo classification report yet at", report_path)


n_train/val/test: 6264 1580 1962

=== comparison_table ===
{
  "No Filter": {
    "precision": 0.5020387359836901,
    "recall": 1.0,
    "f1": 0.668476416694944,
    "accuracy": 0.5020387359836901,
    "fpr": 1.0,
    "rejection_recall": 0.0,
    "rejection_rate": 0.0,
    "tp": 985,
    "tn": 0,
    "fp": 977,
    "fn": 0
  },
  "NLI zero-shot": {
    "precision": 0.8888888888888888,
    "recall": 0.7390862944162436,
    "f1": 0.8070953436807095,
    "accuracy": 0.8226299694189603,
    "fpr": 0.0931422722620266,
    "rejection_recall": 0.9068577277379734,
    "rejection_rate": 0.5825688073394495,
    "tp": 728,
    "tn": 886,
    "fp": 91,
    "fn": 257,
    "threshold": 0.22000000000000003,
    "classification_report": [
      {
        "dataset": "ASQA",
        "num_samples": 1737,
        "accepted": 675,
        "acceptance_rate": 0.38860103626943004,
        "accuracy": 0.8261370178468624,
        "precision": 0.9214814814814815,
        "recall": 0.7141216991963261,
        "f

,dataset,num_samples,accepted,acceptance_rate,accuracy,precision,recall,f1,roc_auc
0,ASQA,1737.0,689.0,0.396661,0.831894,0.920174,0.727899,0.812821,0.949411
1,MS MARCO,207.0,120.0,0.579710,0.845411,0.800000,0.923077,0.857143,0.943521
2,WikiEval,18.0,11.0,0.611111,0.722222,0.727273,0.800000,0.761905,0.850000
3,Overall,1962.0,820.0,0.417941,0.832314,0.900000,0.749239,0.817729,0.936751
